# Lab 08: Linear Attention and Flash Attention

**related Lecture:** Lecture 14

By the end of this lab, you will be able to:

1. Explain why standard self-attention has $O(N^2)$ time and memory complexity.
2. Implement **linear attention** and understand how a kernel feature map turns the cost into $O(N)$.
3. Understand the key ideas behind **FlashAttention**: tiling, online softmax, and IO-awareness.

## Prerequisites

- Basic PyTorch (tensors, `nn.Module`, autograd).
- Familiarity with the Transformer / scaled dot-product attention.
- Big-O notation.

## 0. Setup

make sure to use a colab instance with GPU or TPU.

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 1. Recap: Standard Scaled Dot-Product Attention

Given queries $Q \in \mathbb{R}^{N \times d}$, keys $K \in \mathbb{R}^{N \times d}$, and values $V \in \mathbb{R}^{N \times d}$, scaled dot-product attention is:

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d}}\right) V
$$

### Why is this expensive?

- The matrix $S = Q K^\top$ has shape $N \times N$.
- Computing it costs $O(N^2 d)$ FLOPs.
- Storing it costs $O(N^2)$ memory.

For sequences of length $N = 8{,}192$, that's ~67M entries **per attention head, per layer** — easily many gigabytes for a typical Transformer.

Let's implement it the textbook way.

In [ ]:
# finish implementation of standard_attention

def standard_attention(Q, K, V):
    """Textbook scaled dot-product attention.

    Q, K, V: (B, N, d)
    Returns: (B, N, d)
    """
    d = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d)  # (B, N, N) <-- the bottleneck
    attn =  # complete here
    out =  # complete here
    return out

# Quick sanity check
B, N, d = 2, 16, 8
Q = torch.randn(B, N, d)
K = torch.randn(B, N, d)
V = torch.randn(B, N, d)
out = standard_attention(Q, K, V)
print('Output shape:', out.shape)

### Exercise 1.1

Without running it, predict: if you double $N$, how much more memory does the `scores` tensor use? How much more compute does the matmul require? Write your answer in the cell below.

**Answer:**

## 2. Linear Attention

The core idea of **linear attention** (Katharopoulos et al., 2020) is to replace the softmax with a similarity that **factorizes**.

Recall that softmax attention computes, for each query $i$:

$$
\text{out}_i = \frac{\sum_{j=1}^{N} \text{sim}(q_i, k_j)\, v_j}{\sum_{j=1}^{N} \text{sim}(q_i, k_j)}
$$

where $\text{sim}(q, k) = \exp(q^\top k / \sqrt{d})$.

Suppose we instead use a similarity of the form

$$
\text{sim}(q, k) = \phi(q)^\top \phi(k)
$$

for some feature map $\phi: \mathbb{R}^d \to \mathbb{R}^{d'}$ that produces non-negative features. Then

$$
\text{out}_i = \frac{\phi(q_i)^\top \sum_{j} \phi(k_j) v_j^\top}{\phi(q_i)^\top \sum_{j} \phi(k_j)}
$$

Notice the trick: the sums $\sum_j \phi(k_j) v_j^\top$ and $\sum_j \phi(k_j)$ **do not depend on $i$**. We can compute them **once** and reuse them for every query.

- $S = \sum_j \phi(k_j) v_j^\top \in \mathbb{R}^{d' \times d}$ — a fixed-size matrix!
- $z = \sum_j \phi(k_j) \in \mathbb{R}^{d'}$

The total cost is $O(N d d')$ — **linear in $N$**.

A common, simple choice is $\phi(x) = \text{elu}(x) + 1$, which keeps features non-negative.

In [ ]:
def phi(x):
    """Feature map: elu(x) + 1, ensures non-negativity."""
    return F.elu(x) + 1

def linear_attention(Q, K, V, eps=1e-6):
    """Linear attention with elu+1 feature map.

    Q, K, V: (B, N, d)
    Returns: (B, N, d)
    """
    Qp = phi(Q)  # (B, N, d)
    Kp = phi(K)  # (B, N, d)

    # KV: (B, d, d) -- the 'state' matrix, independent of N in size
    KV = torch.einsum('bnd,bne->bde', Kp, V)

    # Z: (B, d) -- the normalizer
    Z = Kp.sum(dim=1)

    # Numerator: for each query, contract with KV  -> (B, N, d)
    numerator = torch.einsum('bnd,bde->bne', Qp, KV)

    # Denominator: for each query, dot with Z  -> (B, N, 1)
    denominator = torch.einsum('bnd,bd->bn', Qp, Z).unsqueeze(-1) + eps

    return numerator / denominator

# Sanity check shapes
out_lin = linear_attention(Q, K, V)
print('Linear attention output shape:', out_lin.shape)

### Exercise 2.1

Trace through the einsums above on paper. For a batch size of 1 and $N = 4$, $d = 3$, write out the dimensions of `Kp`, `KV`, and `Z`. Confirm the final shape matches the input.

**Answer:**

### How different are the outputs?

Linear attention is **not** an approximation of softmax attention — it's a different function. Let's see how much they differ on random inputs.

In [ ]:
B, N, d = 4, 64, 16
Q = torch.randn(B, N, d)
K = torch.randn(B, N, d)
V = torch.randn(B, N, d)

out_std = standard_attention(Q, K, V)
out_lin = linear_attention(Q, K, V)

diff = (out_std - out_lin).norm() / out_std.norm()
print(f'Relative L2 difference: {diff:.4f}')

### Exercise 2.2

The outputs are clearly different. **Why might linear attention still be a useful drop-in replacement** in a Transformer, even though it doesn't match softmax exactly?

**Answer:**

## 3. FlashAttention

Linear attention changes the **math** to avoid the $N \times N$ matrix. **FlashAttention** (Dao et al., 2022) takes a different approach: it computes the **exact same softmax attention**, but reorganizes the computation to avoid materializing the $N \times N$ matrix in slow GPU memory (HBM).

### The key insight: attention is memory-bound

On modern GPUs, the bottleneck for attention is **not** the FLOPs — it's moving data between:

- **HBM** (high-bandwidth memory): large (40 GB on an A100), but slow (1.5 TB/s).
- **SRAM** (on-chip memory): tiny (192 KB per SM), but very fast (19 TB/s).

Standard attention writes the full $N \times N$ scores to HBM, then reads them back for the softmax, then writes again, then reads again for the matmul with $V$. That's a lot of slow memory traffic for an $N \times N$ matrix.

### FlashAttention's recipe

1. **Tile** $Q$, $K$, $V$ into blocks small enough to fit in SRAM.
2. For each query block, **stream** through the key/value blocks, computing partial attention outputs **without ever storing the full score matrix**.
3. Use the **online softmax** trick to combine partial results correctly.

This gives:

- **Same** numerical result as standard softmax attention (modulo floating-point reordering).
- $O(N^2)$ FLOPs (the math hasn't changed), but only $O(N)$ HBM reads/writes.
- Memory usage drops from $O(N^2)$ to $O(N)$.

### The online softmax trick

Suppose we want $\text{softmax}([s_1, \dots, s_N])$ but we see the values one block at a time. The naive softmax requires knowing all values to find the max (for stability) and the sum of exponentials.

The online version maintains running statistics $(m, \ell)$ where $m$ is the running max and $\ell$ is the running sum of $\exp(s - m)$. When a new block $\{s_k\}$ arrives:

$$
m_{\text{new}} = \max(m, \max_k s_k), \quad \ell_{\text{new}} = e^{m - m_{\text{new}}} \ell + \sum_k e^{s_k - m_{\text{new}}}
$$

Partial outputs are rescaled by $e^{m - m_{\text{new}}}$ to keep things consistent. Let's implement a (slow, pedagogical) Python version.

In [ ]:
def flash_attention_pedagogical(Q, K, V, block_size=16):
    """Pedagogical FlashAttention: same math as softmax attention,
    but computed block by block with online softmax.

    NOTE: This is NOT fast in Python -- the real FlashAttention is a custom
    CUDA kernel. The point here is to show the *algorithm*.
    """
    B, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    O = torch.zeros_like(Q)                     # output
    m = torch.full((B, N, 1), -float('inf'), device=Q.device, dtype=Q.dtype)  # running max
    l = torch.zeros((B, N, 1), device=Q.device, dtype=Q.dtype)               # running sum of exp

    for j in range(0, N, block_size):
        Kj = K[:, j:j+block_size]   # (B, Bk, d)
        Vj = V[:, j:j+block_size]   # (B, Bk, d)

        # Scores for this K-block against ALL queries: (B, N, Bk)
        Sij = torch.matmul(Q, Kj.transpose(-2, -1)) * scale

        # Block-local max
        m_block = Sij.max(dim=-1, keepdim=True).values  # (B, N, 1)

        # New running max
        m_new = torch.maximum(m, m_block)

        # Rescale factors
        alpha = torch.exp(m - m_new)         # rescale previous accumulators

        # exp of scores in new max frame
        Pij = torch.exp(Sij - m_new)         # (B, N, Bk)

        # Update sum and output
        l = alpha * l + Pij.sum(dim=-1, keepdim=True)
        O = alpha * O + torch.matmul(Pij, Vj)
        m = m_new

    return O / l

# Verify: should match standard attention to floating-point precision
B, N, d = 2, 64, 16
Q = torch.randn(B, N, d)
K = torch.randn(B, N, d)
V = torch.randn(B, N, d)

out_std = standard_attention(Q, K, V)
out_flash = flash_attention_pedagogical(Q, K, V, block_size=16)

print(f'Max abs difference: {(out_std - out_flash).abs().max().item():.2e}')
print('(Should be ~1e-6 or smaller -- same math, different reduction order.)')

### Exercise 3.1

Change `block_size` to 1, 8, 32, and 64. Does the output change? Why or why not?

In [ ]:
# Your experiment here

## 4. Empirical comparison

Let's see how the three approaches scale with sequence length.

In [ ]:
def benchmark(fn, Q, K, V, n_warmup=2, n_iter=5):
    for _ in range(n_warmup):
        fn(Q, K, V)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iter):
        fn(Q, K, V)
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iter

B, d = 1, 32
lengths = [256, 512, 1024, 2048, 4096, 8192]

print(f'{"N":>6} | {"standard (ms)":>14} | {"linear (ms)":>12} | {"flash-py (ms)":>14}')
print('-' * 56)
for N in lengths:
    Q = torch.randn(B, N, d, device=device)
    K = torch.randn(B, N, d, device=device)
    V = torch.randn(B, N, d, device=device)

    t_std = benchmark(standard_attention, Q, K, V) * 1000
    t_lin = benchmark(linear_attention, Q, K, V) * 1000
    t_fla = benchmark(lambda q, k, v: flash_attention_pedagogical(q, k, v, block_size=64), Q, K, V) * 1000

    print(f'{N:>6} | {t_std:>14.3f} | {t_lin:>12.3f} | {t_fla:>14.3f}')

### The real FlashAttention

PyTorch ships an optimized fused attention kernel that uses FlashAttention under the hood when conditions are right. Try it:

In [ ]:
# torch.nn.functional.scaled_dot_product_attention dispatches to FlashAttention
# automatically on supported GPUs.
B, H, N, d = 1, 4, 1024, 32
Q = torch.randn(B, H, N, d, device=device)
K = torch.randn(B, H, N, d, device=device)
V = torch.randn(B, H, N, d, device=device)

out = F.scaled_dot_product_attention(Q, K, V)
print('Output shape:', out.shape)



## Further reading

- Katharopoulos et al., *Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention*, 2020.
- Dao et al., *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*, 2022.